# Exploration 03: Early-Layer DiffMean on `CohereLabs/tiny-aya-global` 

Thin Colab/Jupyter wrapper around [`run_tinyaya.sh`](./run_tinyaya.sh) (layers `2, 4, 6, 8, 10, 12`, `alpha=10.0`, full `bfloat16`, 14 high-to-low languages, 70 questions/pair).

> **Before running**: Make sure you have accepted the gated model license on Hugging Face for [`CohereLabs/tiny-aya-global`](https://huggingface.co/CohereLabs/tiny-aya-global) and `uonlp/CulturaX`.

In [ ]:
# 1. Clone / Update Repo
import os

if not os.path.exists('/content/CLaS-Bench'):
    !git clone -b exploration https://github.com/Trier-CLaS-SemEval-2027/CLaS-Bench.git /content/CLaS-Bench

%cd /content/CLaS-Bench
!git checkout exploration
!git pull origin exploration
%cd /content/CLaS-Bench/explorations/03_tiny_aya_early_layers
!pwd

In [ ]:
# 2. Install Dependencies
!pip install -q "vllm==0.10.2" "transformers<5.0.0" datasets fasttext-numpy2-wheel seaborn matplotlib tqdm

In [ ]:
# 3. Check GPU & Login to Hugging Face
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} ({p.total_memory / 1024**3:.1f} GiB)')

from huggingface_hub import login
login()

In [ ]:
# 4. Step 0 (~30s): Smoke Test tiny-aya-global (Cohere2ForCausalLM) in bfloat16
%cd /content/CLaS-Bench/explorations/03_tiny_aya_early_layers
!python 0-check_tinyaya.py

In [ ]:
# 5. Step 1 (One-Time): Tokenize CulturaX & Compute DiffMean Centroids (data_tinyaya/)
%cd /content/CLaS-Bench/explorations/03_tiny_aya_early_layers
!bash run_tinyaya.sh identify

In [ ]:
# 6. Step 2: DiffMean Generation across Early Layers 2, 4, 6, 8, 10, 12 (alpha=10.0, bf16)
%cd /content/CLaS-Bench/explorations/03_tiny_aya_early_layers
!bash run_tinyaya.sh generate

In [ ]:
# 7. Step 3: Evaluate FastText Forcing + Qwen3-8B Semantic Relevance Judge + Plots
%cd /content/CLaS-Bench/explorations/03_tiny_aya_early_layers
!bash run_tinyaya.sh eval

In [ ]:
# 8. Persist Plots & Key Metrics Directly Inside Notebook Output (Safe Against Idle Timeout)
import glob, json
from IPython.display import Image, display

base = '/content/CLaS-Bench/CLaS-Bench/generation/diffmean_tinyaya'
for img in sorted(glob.glob(f'{base}/*.png')):
    print('Plot:', img)
    display(Image(filename=img))

agg = f'{base}/aggregated_results.json'
if os.path.exists(agg):
    print('=== aggregated_results.json ===')
    print(json.dumps(json.load(open(agg)), indent=2))

for p in sorted(glob.glob(f'{base}/layers/*/strength_*/analysis_results.json')):
    d = json.load(open(p))
    print('=== FILE:', p)
    print(json.dumps({'summary': d.get('summary'), 'per_target_language': d.get('per_target_language'), 'language_pairs': d.get('language_pairs')}))